# DiDAQt Receiver & Controller Overhead Benchmark

Measures CPU and memory overhead added by DiDAQt heartbeat processing
to a receiver (scaled by number of senders) and a controller (scaled
by number of receivers and senders per heartbeat).

Creates two FABRIC VMs connected via L2Bridge with ConnectX-6 NICs:
- Node A: traffic generator
- Node B: system under test

In [ ]:
# ---- Configuration (all constants here for notebook resume) ----

SLICE_NAME   = "didaqt_overhead"
NODE_A_NAME  = "traffic_gen"
NODE_B_NAME  = "bench"
CORES        = 8
RAM          = 32
DISK         = 100
IMAGE        = "default_ubuntu_22"

# Receiver benchmark parameters
RX_SENDER_COUNTS = [1, 2, 5, 10, 20, 30, 40, 50]
RX_REPETITIONS   = 5

# Controller benchmark parameters
CTRL_RECEIVER_COUNTS = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
CTRL_SENDERS_PER_HB  = [1, 10, 50]
CTRL_REPETITIONS     = 5

# Network config
NODE_A_IP = "10.0.0.1"
NODE_B_IP = "10.0.0.2"
CTRL_PORT = 9000
DUMMY_PORT = 9999  # receiver heartbeats go here (unused)

# Remote paths
REMOTE_DIR     = "/home/ubuntu/didaqt"
GEN_SCRIPT     = f"{REMOTE_DIR}/benchmarking/gen_overhead_topology.py"
TRAFFIC_GEN    = f"{REMOTE_DIR}/build/overhead_traffic_gen"
OVERHEAD_RX    = f"{REMOTE_DIR}/build/overhead_rx"
HB_GEN         = f"{REMOTE_DIR}/build/overhead_hb_gen"
OVERHEAD_CTRL  = f"{REMOTE_DIR}/build/overhead_ctrl"

# Local paths
LOCAL_REPO    = ".."  # relative to artifact/
LOCAL_RESULTS = "overhead_results"

In [ ]:
# ---- Create FABRIC slice ----

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()

try:
    slice = fablib.get_slice(name=SLICE_NAME)
    print(f"Slice '{SLICE_NAME}' already exists, reusing.")
except:
    site = fablib.get_random_site(
        filter_function=lambda x: x['nic_connectx_6_available'] >= 2
    )
    print(f"Selected site: {site}")

    slice = fablib.new_slice(name=SLICE_NAME)

    node_a = slice.add_node(name=NODE_A_NAME, site=site,
                            cores=CORES, ram=RAM, disk=DISK, image=IMAGE)
    nic_a = node_a.add_component(model='NIC_ConnectX_6', name='nic_a')

    node_b = slice.add_node(name=NODE_B_NAME, site=site,
                            cores=CORES, ram=RAM, disk=DISK, image=IMAGE)
    nic_b = node_b.add_component(model='NIC_ConnectX_6', name='nic_b')

    net = slice.add_l2network(name='data-net', type='L2Bridge')
    iface_a = nic_a.get_interfaces()[0]
    iface_b = nic_b.get_interfaces()[0]
    iface_a.set_mode('config')
    iface_b.set_mode('config')
    net.add_interface(iface_a)
    net.add_interface(iface_b)

    slice.submit()
    print(f"Slice '{SLICE_NAME}' submitted.")

slice.wait_ssh(progress=True)
node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)
print(f"Node A ready: {node_a.get_management_ip()}")
print(f"Node B ready: {node_b.get_management_ip()}")

In [ ]:
# ---- Install dependencies and upload source ----

import os, subprocess, tempfile
from concurrent import futures

node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)

# Install build deps on both nodes
jobs = []
for node in [node_a, node_b]:
    jobs.append(node.execute_thread(
        "sudo apt-get update -qq && "
        "sudo apt-get install -y -qq gcc make libyaml-dev python3"
    ))
for job in futures.as_completed(jobs):
    pass
print("Dependencies installed on both nodes.")

# Create and upload tarball
repo_root = os.path.abspath(LOCAL_REPO)
tarball = os.path.join(tempfile.gettempdir(), "didaqt_src.tar.gz")
subprocess.run(
    ["tar", "czf", tarball,
     "--exclude=.git", "--exclude=build", "--exclude=working",
     "--exclude=artifact/didaqt_experiment.ipynb",
     "--exclude=artifact/didaqt_benchmark.ipynb",
     "--exclude=artifact/didaqt_overhead.ipynb",
     "-C", os.path.dirname(repo_root),
     os.path.basename(repo_root)],
    check=True,
)

jobs = []
for node in [node_a, node_b]:
    node.upload_file(tarball, "/tmp/didaqt_src.tar.gz")
    jobs.append(node.execute_thread(
        f"rm -rf {REMOTE_DIR} && "
        f"tar xzf /tmp/didaqt_src.tar.gz -C /home/ubuntu"
    ))
for job in futures.as_completed(jobs):
    pass
print("Source uploaded to both nodes.")

In [ ]:
# ---- Compile ----

from concurrent import futures

node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)

jobs = []
for node in [node_a, node_b]:
    jobs.append(node.execute_thread(
        f"cd {REMOTE_DIR} && make clean && make && make bench_overhead"
    ))
for job in futures.as_completed(jobs):
    stdout, stderr = job.result()
    if stderr.strip():
        print("STDERR:", stderr)
print("Compiled on both nodes.")

In [ ]:
# ---- Configure network interfaces ----

node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)

iface_a = node_a.get_interface(network_name='data-net')
iface_b = node_b.get_interface(network_name='data-net')

IFACE_A = iface_a.get_physical_os_interface_name()
MAC_A   = iface_a.get_mac()
IFACE_B = iface_b.get_physical_os_interface_name()
MAC_B   = iface_b.get_mac()

print(f"Node A: iface={IFACE_A}  mac={MAC_A}")
print(f"Node B: iface={IFACE_B}  mac={MAC_B}")

# Bring interfaces up and assign IPs
node_a.execute(f"sudo ip link set {IFACE_A} up && "
               f"sudo ip addr add {NODE_A_IP}/24 dev {IFACE_A}")
node_b.execute(f"sudo ip link set {IFACE_B} up && "
               f"sudo ip addr add {NODE_B_IP}/24 dev {IFACE_B}")

# Verify connectivity
stdout, _ = node_a.execute(f"ping -c 2 {NODE_B_IP}")
print(stdout)

In [ ]:
# ---- Run receiver overhead benchmark ----

import time

node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)

rx_rows = []  # (num_senders, mode, cpu_user_us, cpu_sys_us, peak_rss_kb, frames, rep)

for ns in RX_SENDER_COUNTS:
    print(f"\n--- Receiver benchmark: {ns} senders ---")
    for rep in range(RX_REPETITIONS):
        for mode in ['baseline', 'didaqt']:
            # Start traffic generator on Node A (background)
            node_a.execute_thread(
                f"sudo {TRAFFIC_GEN} {IFACE_A} {MAC_B} {ns} &"
            )
            time.sleep(1)  # let traffic gen start

            # Run measurement on Node B
            stdout, stderr = node_b.execute(
                f"sudo {OVERHEAD_RX} {IFACE_B} {ns} {NODE_A_IP} {DUMMY_PORT} {mode}",
                quiet=True,
            )

            # Kill traffic gen
            node_a.execute("sudo pkill -f overhead_traffic_gen || true",
                           quiet=True)

            # Parse output
            line = stdout.strip()
            if not line:
                print(f"  rep {rep} {mode}: no output")
                if stderr.strip():
                    print(f"  stderr: {stderr.strip()}")
                continue

            parts = line.split(',')
            rx_rows.append((
                int(parts[0]),    # num_senders
                parts[1],         # mode
                int(parts[2]),    # cpu_user_us
                int(parts[3]),    # cpu_sys_us
                int(parts[4]),    # peak_rss_kb
                int(parts[5]),    # frames_received
                rep,
            ))

        cpu_d = [r for r in rx_rows if r[0] == ns and r[1] == 'didaqt']
        cpu_b = [r for r in rx_rows if r[0] == ns and r[1] == 'baseline']
        if cpu_d and cpu_b:
            d_cpu = (cpu_d[-1][2] + cpu_d[-1][3]) / 1e6 * 100 / 10
            b_cpu = (cpu_b[-1][2] + cpu_b[-1][3]) / 1e6 * 100 / 10
            print(f"  rep {rep}: didaqt={d_cpu:.2f}% baseline={b_cpu:.2f}% "
                  f"frames={cpu_d[-1][5]}")

print("\n=== Receiver benchmark complete ===")

In [ ]:
# ---- Run controller overhead benchmark ----

import time

node_a = slice.get_node(name=NODE_A_NAME)
node_b = slice.get_node(name=NODE_B_NAME)

READY_FILE = "/tmp/ctrl_ready"

def wait_for_ready(node, path, timeout=300, poll_interval=1):
    """Poll for a marker file on a remote node."""
    elapsed = 0
    while elapsed < timeout:
        stdout, _ = node.execute(f"test -f {path} && echo YES || echo NO",
                                 quiet=True)
        if stdout.strip() == "YES":
            return True
        time.sleep(poll_interval)
        elapsed += poll_interval
    return False

ctrl_rows = []  # (num_recv, senders_per_hb, mode, cpu_user, cpu_sys, rss, pkts, rep)

for nr in CTRL_RECEIVER_COUNTS:
    for sph in CTRL_SENDERS_PER_HB:
        label = f"r{nr}_s{sph}"
        topo_file = f"/tmp/{label}.yaml"

        print(f"\n--- Controller benchmark: {nr} receivers, {sph} senders/hb ---")

        # Generate topology on Node B
        print(f"  generating topology...")
        stdout, stderr = node_b.execute(
            f"python3 {GEN_SCRIPT} {nr} {sph} {topo_file}",
            quiet=True,
        )
        if stderr.strip():
            print(f"  gen stderr: {stderr.strip()}")

        for rep in range(CTRL_REPETITIONS):
            for mode in ['baseline', 'didaqt']:
                # Remove stale ready marker
                node_b.execute(f"rm -f {READY_FILE}", quiet=True)

                # Start controller on Node B (background)
                ctrl_job = node_b.execute_thread(
                    f"{OVERHEAD_CTRL} {CTRL_PORT} {topo_file} {nr} {sph} {mode} {READY_FILE}"
                )

                # Wait for controller to signal readiness
                if not wait_for_ready(node_b, READY_FILE):
                    print(f"  rep {rep} {mode}: controller failed to become ready")
                    node_a.execute("pkill -f overhead_hb_gen || true", quiet=True)
                    try:
                        ctrl_job.result()
                    except:
                        pass
                    continue

                # Start heartbeat generator on Node A
                # Use data-plane IP (NODE_B_IP) — management network blocks UDP
                hb_job = node_a.execute_thread(
                    f"timeout 30 {HB_GEN} {NODE_B_IP} {CTRL_PORT} {nr} {sph}"
                )

                # Wait for controller to finish (warmup + measure + margin)
                try:
                    stdout, stderr = ctrl_job.result()
                except Exception as e:
                    print(f"  rep {rep} {mode}: controller error: {e}")
                    node_a.execute("pkill -f overhead_hb_gen || true", quiet=True)
                    continue

                # Kill heartbeat gen
                node_a.execute("pkill -f overhead_hb_gen || true", quiet=True)

                line = stdout.strip().split('\n')[-1]  # last line is CSV
                if not line or ',' not in line:
                    print(f"  rep {rep} {mode}: no output")
                    if stderr.strip():
                        print(f"  stderr: {stderr.strip()}")
                    continue

                parts = line.split(',')
                ctrl_rows.append((
                    int(parts[0]),    # num_receivers
                    int(parts[1]),    # senders_per_hb
                    parts[2],         # mode
                    int(parts[3]),    # cpu_user_us
                    int(parts[4]),    # cpu_sys_us
                    int(parts[5]),    # peak_rss_kb
                    int(parts[6]),    # packets_received
                    rep,
                ))

            d = [r for r in ctrl_rows if r[0]==nr and r[1]==sph and r[2]=='didaqt']
            b = [r for r in ctrl_rows if r[0]==nr and r[1]==sph and r[2]=='baseline']
            if d and b:
                d_cpu = (d[-1][3] + d[-1][4]) / 1e6 * 100 / 10
                b_cpu = (b[-1][3] + b[-1][4]) / 1e6 * 100 / 10
                print(f"  rep {rep}: didaqt={d_cpu:.2f}% baseline={b_cpu:.2f}% "
                      f"rss_d={d[-1][5]}KB rss_b={b[-1][5]}KB pkts={d[-1][6]}")

        # Clean up topology file
        node_b.execute(f"rm -f {topo_file}", quiet=True)

print("\n=== Controller benchmark complete ===")

In [ ]:
# ---- Save results to CSV ----

import os, csv

os.makedirs(LOCAL_RESULTS, exist_ok=True)

rx_csv = os.path.join(LOCAL_RESULTS, "rx_overhead.csv")
with open(rx_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["num_senders", "mode", "cpu_user_us", "cpu_sys_us",
                "peak_rss_kb", "frames_received", "repetition"])
    for row in rx_rows:
        w.writerow(row)
print(f"Wrote {rx_csv} ({len(rx_rows)} rows)")

ctrl_csv = os.path.join(LOCAL_RESULTS, "ctrl_overhead.csv")
with open(ctrl_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["num_receivers", "senders_per_hb", "mode", "cpu_user_us",
                "cpu_sys_us", "peak_rss_kb", "packets_received", "repetition"])
    for row in ctrl_rows:
        w.writerow(row)
print(f"Wrote {ctrl_csv} ({len(ctrl_rows)} rows)")

In [ ]:
# ---- Plot results ----

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

rx_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "rx_overhead.csv"))
ctrl_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "ctrl_overhead.csv"))

# Compute CPU % = (user + sys) / (10s * 1e6 us) * 100
MEASURE_US = 10 * 1e6
rx_df['cpu_pct'] = (rx_df['cpu_user_us'] + rx_df['cpu_sys_us']) / MEASURE_US * 100
ctrl_df['cpu_pct'] = (ctrl_df['cpu_user_us'] + ctrl_df['cpu_sys_us']) / MEASURE_US * 100

# IEEE dual-column style
COL_WIDTH = 3.4
COL_HEIGHT = 2.4

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "lines.markersize": 4,
    "lines.linewidth": 1,
})

RX_STYLES = {
    'didaqt':   {"color": "#377eb8", "marker": "o", "linestyle": "-"},
    'baseline': {"color": "#999999", "marker": "s", "linestyle": "--"},
}

# Controller styles: same color/marker per senders_per_hb,
# solid for didaqt, dotted for baseline
CTRL_COLORS = {
    1:  {"color": "#377eb8", "marker": "o"},
    10: {"color": "#ff7f00", "marker": "s"},
    50: {"color": "#4daf4a", "marker": "^"},
}

def sph_label(s):
    return f"{s} sender{'s' if s > 1 else ''}/hb"

# ---- Graph 1: Receiver CPU Overhead ----
rx_cpu = rx_df.groupby(['num_senders', 'mode'])['cpu_pct'].median().reset_index()

fig1, ax1 = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for mode in ['didaqt', 'baseline']:
    sub = rx_cpu[rx_cpu['mode'] == mode].sort_values('num_senders')
    label = 'With DiDAQt' if mode == 'didaqt' else 'Baseline'
    ax1.plot(sub['num_senders'], sub['cpu_pct'],
             label=label, **RX_STYLES[mode])
ax1.set_xlabel('Number of senders')
ax1.set_ylabel('CPU usage (%)')
ax1.legend()
ax1.grid(True, which='both', ls=':', lw=0.5)
fig1.tight_layout()
fig1.savefig(os.path.join(LOCAL_RESULTS, 'rx_cpu_overhead.pdf'), bbox_inches='tight')
plt.show()

# ---- Graph 2: Receiver Memory Overhead ----
rx_mem = rx_df.groupby(['num_senders', 'mode'])['peak_rss_kb'].median().reset_index()

fig2, ax2 = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for mode in ['didaqt', 'baseline']:
    sub = rx_mem[rx_mem['mode'] == mode].sort_values('num_senders')
    label = 'With DiDAQt' if mode == 'didaqt' else 'Baseline'
    ax2.plot(sub['num_senders'], sub['peak_rss_kb'],
             label=label, **RX_STYLES[mode])
ax2.set_xlabel('Number of senders')
ax2.set_ylabel('Peak RSS (KB)')
ax2.legend()
ax2.grid(True, which='both', ls=':', lw=0.5)
fig2.tight_layout()
fig2.savefig(os.path.join(LOCAL_RESULTS, 'rx_memory_overhead.pdf'), bbox_inches='tight')
plt.show()

# ---- Graph 3: Controller CPU Overhead ----
ctrl_cpu = ctrl_df.groupby(
    ['num_receivers', 'senders_per_hb', 'mode'])['cpu_pct'].median().reset_index()

fig3, ax3 = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sph in sorted(ctrl_cpu['senders_per_hb'].unique()):
    for mode in ['didaqt', 'baseline']:
        sub = ctrl_cpu[(ctrl_cpu['senders_per_hb'] == sph) &
                       (ctrl_cpu['mode'] == mode)].sort_values('num_receivers')
        ls = '-' if mode == 'didaqt' else ':'
        label = sph_label(sph) if mode == 'didaqt' else f"{sph_label(sph)} (baseline)"
        ax3.plot(sub['num_receivers'], sub['cpu_pct'],
                 linestyle=ls, label=label, **CTRL_COLORS[sph])
ax3.set_xscale('log')
ax3.set_xlabel('Number of receivers')
ax3.set_ylabel('CPU usage (%)')
ax3.legend()
ax3.grid(True, which='both', ls=':', lw=0.5)
fig3.tight_layout()
fig3.savefig(os.path.join(LOCAL_RESULTS, 'ctrl_cpu_overhead.pdf'), bbox_inches='tight')
plt.show()

# ---- Graph 4: Controller Memory Overhead ----
ctrl_mem = ctrl_df.groupby(
    ['num_receivers', 'senders_per_hb', 'mode'])['peak_rss_kb'].median().reset_index()

fig4, ax4 = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sph in sorted(ctrl_mem['senders_per_hb'].unique()):
    for mode in ['didaqt', 'baseline']:
        sub = ctrl_mem[(ctrl_mem['senders_per_hb'] == sph) &
                       (ctrl_mem['mode'] == mode)].sort_values('num_receivers')
        ls = '-' if mode == 'didaqt' else ':'
        label = sph_label(sph) if mode == 'didaqt' else f"{sph_label(sph)} (baseline)"
        ax4.plot(sub['num_receivers'], sub['peak_rss_kb'] / 1024,
                 linestyle=ls, label=label, **CTRL_COLORS[sph])
ax4.set_xscale('log')
ax4.set_yscale('log')
ax4.set_xlabel('Number of receivers')
ax4.set_ylabel('Peak RSS (MB)')
ax4.legend()
ax4.grid(True, which='both', ls=':', lw=0.5)
fig4.tight_layout()
fig4.savefig(os.path.join(LOCAL_RESULTS, 'ctrl_memory_overhead.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# ---- Cleanup (optional) ----

# slice.delete()